# 01 · Coleta e validação dos dados

**Pergunta do projeto:** a demanda por trabalho em tecnologia vai aumentar ou diminuir nos próximos 10 anos?

Este notebook baixa **apenas dados oficiais e públicos**, salva um cache em `data/raw/` e gera
tabelas limpas em `data/processed/` para os notebooks seguintes.

| Fonte | O que mede | Cobertura |
|---|---|---|
| **BLS / CES via FRED** | Emprego mensal por setor (milhares) | EUA, 1990 → hoje |
| **BLS / JOLTS via FRED** | Vagas abertas no setor de Informação | EUA, 2000 → hoje |
| **Indeed Hiring Lab** | Índice diário de anúncios de vaga por área (fev/2020 = 100) | 6 países, 2020 → hoje |
| **Eurostat** (`isoc_sks_itspt`) | Especialistas em TIC empregados | UE + 36 países, 2004 → hoje |
| **BLS Employment Projections** | Projeção oficial de emprego por ocupação | EUA, 2025 → 2035 |

In [1]:
import pandas as pd
from techdemand import config, data_sources as ds

pd.set_option("display.float_format", "{:,.1f}".format)

## 1. Séries de emprego e vagas (FRED / BLS)

In [2]:
fred = ds.fetch_fred_panel()
catalogo = pd.DataFrame({
    "descrição": config.FRED_SERIES,
    "início": fred.apply(lambda s: s.first_valid_index().date()),
    "fim": fred.apply(lambda s: s.last_valid_index().date()),
    "meses": fred.count(),
    "último valor": fred.apply(lambda s: s.dropna().iloc[-1]),
})
catalogo

,descrição,início,fim,meses,último valor
CES6054150001,Design de sistemas de computação e serviços re...,1990-01-01,2026-09-01,441,"2,355.4"
CES5051800001,"Infraestrutura de computação, processamento de...",1990-01-01,2026-09-01,441,453.8
CES6054160001,Consultoria de gestão e técnica,1990-01-01,2026-09-01,441,"1,888.1"
CES5051700001,Telecomunicações,1990-01-01,2026-09-01,441,564.6
USINFO,Setor de Informação (total),1939-01-01,2026-09-01,1053,"2,739.0"
PAYEMS,Emprego total não agrícola (EUA),1939-01-01,2026-09-01,1053,"159,044.0"
JTU5100JOL,"Vagas abertas no setor de Informação (JOLTS, s...",2000-12-01,2026-08-01,309,128.0
IHLIDXUS,"Indeed: índice de vagas, total (fev/2020 = 100)",2020-02-01,2026-10-01,81,104.0
IHLIDXUSTPSOFTDEVE,"Indeed: índice de vagas, desenvolvimento de so...",2020-02-01,2026-10-01,81,78.3
IHLIDXUSTPMATH,"Indeed: índice de vagas, matemática",2020-02-01,2026-10-01,81,63.8


## 2. Anúncios de vagas por área — Indeed Hiring Lab (10 países)

In [3]:
indeed = ds.fetch_indeed_all_countries()
indeed = indeed[indeed["variable"] == "total postings"]
print(f"{len(indeed):,} linhas · {indeed.sector.nunique()} áreas · {indeed.jobcountry.nunique()} países")
print("Período:", indeed.date.min().date(), "→", indeed.date.max().date())
indeed[indeed.sector.isin(config.TECH_SECTORS)].groupby(["jobcountry", "sector"])["index"].last().unstack().round(1)

659,050 linhas · 53 áreas · 6 países
Período: 2020-02-01 → 2026-10-16


sector,Data & Analytics,"IT Infrastructure, Operations & Support",IT Systems & Solutions,Software Development
jobcountry,,,,
AU,72.2,112.9,108.4,105.7
CA,87.3,66.1,85.6,75.4
DE,72.5,63.3,79.2,47.7
FR,NaN,62.6,62.5,51.3
GB,49.8,46.1,62.0,60.0
US,63.6,68.5,77.6,78.9


## 3. Especialistas em TIC na Europa — Eurostat

In [4]:
eu = ds.fetch_eurostat_ict()
print(eu.geo.nunique(), "países/agrupamentos ·", eu.time.min(), "→", eu.time.max())
eu[eu.geo == "EU27_2020"].pivot(index="time", columns="unit", values="value").tail(8)

37 países/agrupamentos · 2004 → 2025


unit,PC_EMP,THS_PER
time,,
2018,3.8,"7,570.4"
2019,4.0,"7,880.4"
2020,4.3,"8,442.4"
2021,4.5,"8,920.3"
2022,4.6,"9,370.1"
2023,4.8,"9,749.5"
2024,4.9,"10,189.9"
2025,5.0,"10,450.8"


## 4. Projeções oficiais BLS 2025–2035 (snapshot validado por testes)

In [5]:
bls = ds.load_bls_projections()
bls[bls.level.isin(["total", "major", "minor"])]

,soc_code,title,level,employment_2025,employment_2035,change_pct,annual_openings,median_wage_2025
0,00-0000,"Total, all occupations",total,"170,280.8","176,198.5",3.5,"17,461.8",50980
1,15-0000,Computer and mathematical occupations,major,"5,493.6","5,896.7",7.3,316.1,109280
2,15-1200,Computer occupations,minor,"5,036.2","5,323.8",5.7,280.0,109470
22,15-2000,Mathematical science occupations,minor,457.4,572.9,25.3,36.1,107570


## 5. Qualidade dos dados

Checagens objetivas antes de qualquer análise: valores ausentes, duplicatas e valores impossíveis.

In [6]:
qualidade = pd.DataFrame({
    "linhas": [len(fred), len(indeed), len(eu), len(bls)],
    "nulos (%)": [fred.isna().mean().mean() * 100, indeed["index"].isna().mean() * 100,
                  eu["value"].isna().mean() * 100, bls.isna().mean().mean() * 100],
    "duplicatas": [fred.index.duplicated().sum(),
                   indeed.duplicated(["date", "jobcountry", "sector"]).sum(),
                   eu.duplicated(["unit", "geo", "time"]).sum(),
                   bls.soc_code.duplicated().sum()],
    "valores ≤ 0": [(fred <= 0).sum().sum(), (indeed["index"] <= 0).sum(),
                    (eu["value"] < 0).sum(), (bls.employment_2025 <= 0).sum()],
}, index=["FRED", "Indeed", "Eurostat", "BLS projeções"])
assert qualidade["duplicatas"].sum() == 0
qualidade

,linhas,nulos (%),duplicatas,valores ≤ 0
FRED,1054,58.0,0,0
Indeed,659050,0.0,0,0
Eurostat,1452,0.0,0,0
BLS projeções,29,0.0,0,0


> O painel FRED está em frequência mensal. Seus nulos vêm apenas das datas de início diferentes entre séries
> (ex.: o setor de Informação começa em 1939, o Indeed só em 2020), não de falhas de coleta.

## 6. Exportação dos dados processados

In [7]:
config.DATA_PROCESSED.mkdir(parents=True, exist_ok=True)
fred.to_csv(config.DATA_PROCESSED / "fred_panel.csv")
indeed.to_csv(config.DATA_PROCESSED / "indeed_postings.csv", index=False)
eu.to_csv(config.DATA_PROCESSED / "eurostat_ict.csv", index=False)
sorted(p.name for p in config.DATA_PROCESSED.glob("*.csv"))

['eurostat_ict.csv', 'fred_panel.csv', 'indeed_postings.csv']